<a href="https://colab.research.google.com/github/Anusha-Antony/Agentic_kerala-feed/blob/main/mini_pro_kerala.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# Cell 1 — Install dependencies
!pip install groq wikipedia gradio --quiet

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.7/143.7 kB 3.7 MB/s eta 0:00:00


In [2]:
!pip install wikipedia-api
import random
import wikipediaapi
import re


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.8/43.8 kB 1.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 129.2/129.2 kB 4.5 MB/s eta 0:00:00


In [3]:

!pip install -U gradio gradio-client

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 31.3/31.3 MB 59.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.6/73.6 kB 5.9 MB/s eta 0:00:00
  Attempting uninstall: starlette
    Found existing installation: starlette 0.52.1
    Uninstalling starlette-0.52.1:
      Successfully uninstalled starlette-0.52.1
  Attempting uninstall: gradio-client
    Found existing installation: gradio_client 1.14.0
    Uninstalling gradio_client-1.14.0:
      Successfully uninstalled gradio_client-1.14.0
  Attempting uninstall: gradio
    Found existing installation: gradio 5.50.0
    Uninstalling gradio-5.50.0:
      Successfully uninstalled gradio-5.50.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-adk 1.29.0 requires starlette<1.0.0,>=0.49.1, but you have starlette 1.3.1 whi

In [4]:
# Cell 2 — Imports and API key setup
# ⚠️  NEVER hardcode your API key. Use Colab Secrets:
#   Runtime → Manage secrets → Add secret named GROQ_API_KEY

import os
import re
import random
import wikipedia
import gradio as gr
from groq import Groq

# Load API key from Colab Secrets
try:
    from google.colab import userdata
    GROQ_API_KEY = userdata.get('groq_key')
except Exception:
    # Fallback: read from environment variable
    GROQ_API_KEY = os.environ.get('GROQ_API_KEY', '')

if not GROQ_API_KEY:
    raise ValueError("❌ GROQ_API_KEY not found. Add it via Runtime → Manage secrets.")

client = Groq(api_key=GROQ_API_KEY)
print("✅ Groq client initialised successfully.")

✅ Groq client initialised successfully.


In [5]:
!pip install wikipedia-api
import random
import wikipediaapi
import re


In [6]:
KNOWLEDGE_BASE = []

wiki = wikipediaapi.Wikipedia(
    language="en",
    user_agent="KeralaHistoryFeed/1.0 (colab-project; educational use)"
)

def clean_text(text: str) -> str:
    text = re.sub(r"=+[^=]+=+", "", text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()

def chunk_text(text: str, chunk_size: int = 350) -> list:
    words = text.split()
    return [
        " ".join(words[i : i + chunk_size])
        for i in range(0, len(words), chunk_size)
        if words[i : i + chunk_size]
    ]

def load_knowledge_base(page_titles: list) -> str:
    global KNOWLEDGE_BASE
    KNOWLEDGE_BASE = []
    loaded, failed = [], []
    for title in page_titles:
        try:
            page = wiki.page(title)
            if not page.exists():
                failed.append(f"{title} (not found)")
                continue
            cleaned = clean_text(page.text)
            if len(cleaned.split()) < 50:
                failed.append(f"{title} (too short)")
                continue
            for chunk in chunk_text(cleaned):
                KNOWLEDGE_BASE.append({"chunk": chunk, "source": page.title})
            loaded.append(title)
        except Exception as e:
            failed.append(f"{title} ({e})")

    status = f"✅ Loaded {len(loaded)} pages → {len(KNOWLEDGE_BASE)} chunks."
    if failed:
        status += f"  ⚠️ Skipped: {', '.join(failed)}"
    return status

print("✅ Knowledge base ready.")

✅ Knowledge base ready.


In [11]:
# -----------------------------------------------------------------------
# Topic Presets
# -----------------------------------------------------------------------

TOPIC_PRESETS = {
    "🏛️ History of Kerala": [
        "History of Kerala",
        "Kerala",
        "Chera dynasty",
        "Kingdom of Travancore",
        "Kingdom of Cochin",
        "Malabar",
        "Muziris",
        "Spice trade",
        "Culture of Kerala"
    ]
}

In [12]:
import gradio as gr

def handle_load_topic(topic_name):
    pages = TOPIC_PRESETS.get(
        topic_name,
        TOPIC_PRESETS["🏛️ History of Kerala"]
    )
    return load_knowledge_base(pages)


def handle_next_card():
    return generate_card()


with gr.Blocks(title="Kerala Feed") as demo:

    gr.HTML("""
    <div style="text-align:center;padding:32px 0 16px;">
        <div style="
            font-family:Georgia,serif;
            font-size:2.4rem;
            font-weight:700;
            color:#f5c97a;
            letter-spacing:6px;
            margin:0;
            text-shadow:0 0 24px #c8860099;">
            ✦ KERALA FEED
        </div>
    </div>
    """)

    with gr.Row():
        topic_dd = gr.Dropdown(
            choices=list(TOPIC_PRESETS.keys()),
            value="🏛️ History of Kerala",
            label="Select Topic",
            scale=4
        )

        load_btn = gr.Button(
            "⟳ LOAD TOPIC",
            elem_id="load-btn",
            scale=1
        )

    status_box = gr.Textbox(
        value="← Select a topic and click LOAD TOPIC to begin.",
        label="Status",
        interactive=False,
        elem_id="status-box",
        lines=1
    )

    card_out = gr.Textbox(
        value="Click ▶ NEXT CARD after loading a topic to see your first fact.",
        label="Fact Card",
        interactive=False,
        elem_id="card-output",
        lines=6
    )

    next_btn = gr.Button(
        "▶ NEXT CARD",
        elem_id="next-btn"
    )

    load_btn.click(
        fn=handle_load_topic,
        inputs=topic_dd,
        outputs=status_box
    )

    next_btn.click(
        fn=handle_next_card,
        outputs=card_out
    )

demo.launch()

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://57dfdcad8ab957cf52.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [10]:
import gradio as gr

def handle_load_topic(topic_name):
    pages = TOPIC_PRESETS.get(
        topic_name,
        TOPIC_PRESETS["🏛️ History of Kerala"]
    )
    return load_knowledge_base(pages)


def handle_next_card():
    return generate_card()


with gr.Blocks(title="Kerala Feed") as demo:

    gr.HTML("""
    <div style="text-align:center;padding:32px 0 16px;">
        <div style="
            font-family:Georgia,serif;
            font-size:2.4rem;
            font-weight:700;
            color:#f5c97a;
            letter-spacing:6px;
            margin:0;
            text-shadow:0 0 24px #c8860099;">
            ✦ KERALA FEED
        </div>
    </div>
    """)

    with gr.Row():
        topic_dd = gr.Dropdown(
            choices=list(TOPIC_PRESETS.keys()),
            value="🏛️ History of Kerala",
            label="Select Topic",
            scale=4
        )

        load_btn = gr.Button(
            "⟳ LOAD TOPIC",
            elem_id="load-btn",
            scale=1
        )

    status_box = gr.Textbox(
        value="← Select a topic and click LOAD TOPIC to begin.",
        label="Status",
        interactive=False,
        elem_id="status-box",
        lines=1
    )

    card_out = gr.Textbox(
        value="Click ▶ NEXT CARD after loading a topic to see your first fact.",
        label="Fact Card",
        interactive=False,
        elem_id="card-output",
        lines=6
    )

    next_btn = gr.Button(
        "▶ NEXT CARD",
        elem_id="next-btn"
    )

    load_btn.click(
        fn=handle_load_topic,
        inputs=topic_dd,
        outputs=status_box
    )

    next_btn.click(
        fn=handle_next_card,
        outputs=card_out
    )

print("✅ Gradio interface built.")

NameError: name 'TOPIC_PRESETS' is not defined

In [ ]:
demo.launch(share=True)